In [ ]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents


reader = GithubRepositoryDataReader(
    repo_owner="spring-projects",
    repo_name="spring-ai",
    commit_id="main",             # or a specific commit SHA / tag like your example
    allowed_extensions={"adoc"},   # Spring AI docs use .adoc, not .md
    filename_filter=lambda
    path: "/pages/" in path,
)
files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

chunks = chunk_documents(documents, size=2000, step=1000)

print(chunks[2])

print(f"Content: {chunks[2]['content']}")
print(f"Filename: {chunks[2]['filename']}")
print(f"Start: {chunks[2]['start']}")



In [ ]:
from elasticsearch import Elasticsearch
    
es = Elasticsearch("http://localhost:9200")

response = es.search(
    index="spring-ai-docs",
    query={"match_all": {}},
    size=5,
    source_excludes=["embedding"],
)

for hit in response["hits"]["hits"]:
    print(hit["_source"])

In [7]:
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer



query = "How can I unit test spring ai?"

es = Elasticsearch("http://localhost:9200")

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()


# BM25 search
bm25 = es.search(
        index="spring-ai-docs",
        query={
            "match": {
                "content": query
            }
        },
        size=20,
        source_excludes=["embedding"],
    )

# Vector search
knn = es.search(
        index="spring-ai-docs",
        knn={
            "field": "embedding",
            "query_vector": query_embedding,
            "k": 20,
            "num_candidates": 100,
        },
        source_excludes=["embedding"],
    )

scores = {}
docs = {}
for ranking in [bm25["hits"]["hits"], knn["hits"]["hits"]]:
    for rank, hit in enumerate(ranking):
        doc_id = hit["_id"]
        docs[doc_id] = hit
        scores.setdefault(doc_id, 0.0)
        scores[doc_id] += 1.0 / (60 + rank + 1)
ranked = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True,
)

print("Ranked results:")
documents =[
        docs[doc_id]["_source"]
        for doc_id, _ in ranked[:5]
    ]

context = "\n\n---\n\n".join(
        f"Source: {doc['filename']}\n\n{doc['content']}"
        for doc in documents
    )

system_prompt = """You are an expert assistant for the Spring AI documentation.

Answer questions using ONLY the provided documentation.

Rules:
- If the answer is in the documentation, answer clearly and concisely.
- If the documentation does not contain the answer, say:
  "I couldn't find that information in the Spring AI documentation."
- Do not invent or assume information.
- When appropriate, mention which document the information came from.
"""

user_prompt = f"""Documentation:

{context}

Question:
{query}
"""

prompts=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ranked results:


In [8]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-5-mini",
    input=prompts,
)

print("Response from GPT-5-mini:")
print(response.output_text)

Response from GPT-5-mini:
You can use the Spring AI Testcontainers support to run model services or vector stores in tests. From the documentation (testcontainers.adoc):

- Add the testcontainers auto-configuration dependency:
  - Maven:
    <dependency>
       <groupId>org.springframework.ai</groupId>
       <artifactId>spring-ai-spring-boot-testcontainers</artifactId>
    </dependency>
  - Gradle:
    dependencies {
        implementation 'org.springframework.ai:spring-ai-spring-boot-testcontainers'
    }

- The module provides built-in service connection factories for containers such as:
  - AwsOpenSearchConnectionDetails — LocalStackContainer
  - ChromaConnectionDetails — ChromaDBContainer
  - McpSseClientConnectionDetails — DockerMcpGatewayContainer
  - MilvusServiceClientConnectionDetails — MilvusContainer
  - OllamaConnectionDetails — OllamaContainer
  - OpenSearchConnectionDetails — OpenSearchContainer
  - QdrantConnectionDetails — QdrantContainer
  - TypesenseConnectionDetails